## Codigo para crear test.npz de un archivo .nc

### con netcdf4


In [3]:
import pandas as pd
import numpy as np
from netCDF4 import Dataset as netcdf       # netcdf4-python module
from netCDF4 import num2date  
# https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html
a = netcdf('/tf/workspace/sst.mnmean_12_2025.nc') # 2 grados
a
sst = a.variables['sst'][:] # desde 1854/1 - 2025/12  = 2064 meses
time_sst = a.variables['time']
lon = a.variables['lon'][:]
lat = a.variables['lat'][:]

In [ ]:
# GUARDANDO EL ARCHIVO NETCDF CON NPZ

# guardando 'masked array'como array test.npz para su corrida en tensorflow :
#np.savez_compressed('test2g_12_2025.npz',
#                    data=sst.data,
#                    mask=sst.mask)

# Guardar de forma eficiente
np.savez_compressed(
    'test2g_12_2025.npz',
    data=sst.data,      # Datos numéricos
    mask=sst.mask,      # Máscara booleana
    time=time_sst,      # Metadatos importantes
    lat=lat,
    lon=lon
)

a.close()
print("✓ Archivo NPZ creado exitosamente")


✓ Archivo NPZ creado exitosamente


In [4]:
# Cómo cargarlo después
loaded = np.load("test2g_12_2025.npz")

sst = np.ma.MaskedArray(
    loaded["data"],   # <- coincide con el nombre en el NPZ
    mask=loaded["mask"]
)

time = loaded["time"]
lat = loaded["lat"]
lon = loaded["lon"]

print(sst.shape)
print(time.shape, lat.shape, lon.shape)


(2064, 89, 180)
(2064,) (89,) (180,)


In [ ]:
sstnp = np.array(sst)
sstnp

pdsst = pd.DataFrame(sstnp[0])
#print(pdsst)   

# Guardando datos en csv
#pdsst.to_csv('datos_guardados.csv', index=False)

total = (pdsst == -9.96921e+36).sum().sum()
print(f"El total de veces que aparece -9.96921e+36 (NaN) por mes, es: {total}")

El total de veces que aparece -9.96921e+36 (NaN) por mes, es: 5032


### Usando xarray

In [ ]:
import xarray as xr

ds = xr.open_dataset('/tf/workspace/sst.mnmean_12_2025.nc') # desde 1854/1 - 2025/12  = 2064 meses
print(ds)


<xarray.Dataset> Size: 132MB
Dimensions:    (time: 2064, nbnds: 2, lat: 89, lon: 180)
Coordinates:
  * time       (time) datetime64[ns] 17kB 1854-01-01 1854-02-01 ... 2025-12-01
  * lat        (lat) float32 356B 88.0 86.0 84.0 82.0 ... -84.0 -86.0 -88.0
  * lon        (lon) float32 720B 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
Dimensions without coordinates: nbnds
Data variables:
    time_bnds  (time, nbnds) float64 33kB ...
    sst        (time, lat, lon) float32 132MB ...
Attributes: (12/37)
    climatology:               Climatology is based on 1971-2000 SST, Xue, Y....
    description:               In situ data: ICOADS2.5 before 2007 and NCEP i...
    keywords_vocabulary:       NASA Global Change Master Directory (GCMD) Sci...
    keywords:                  Earth Science > Oceans > Ocean Temperature > S...
    instrument:                Conventional thermometers
    source_comment:            SSTs were observed by conventional thermometer...
    ...                        ...
 

In [28]:
# Extraer variables 
print("------------ SST ------------------------------")
sst = ds["sst"]   # cambia por el nombre real
print(sst)
print("------------ TIME -----------------------------")
time_sst = ds["time"]
print(time_sst)
print("------------ LAT ------------------------------")
lat = ds["lat"]
print(lat)
print("------------ LON ------------------------------")
lon = ds["lon"]
print(lon)


------------ SST ------------------------------
<xarray.DataArray 'sst' (time: 2064, lat: 89, lon: 180)> Size: 132MB
[33065280 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 17kB 1854-01-01 1854-02-01 ... 2025-12-01
  * lat      (lat) float32 356B 88.0 86.0 84.0 82.0 ... -82.0 -84.0 -86.0 -88.0
  * lon      (lon) float32 720B 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
Attributes:
    long_name:     Monthly Means of Sea Surface Temperature
    units:         degC
    var_desc:      Sea Surface Temperature
    level_desc:    Surface
    statistic:     Mean
    dataset:       NOAA Extended Reconstructed SST V5
    parent_stat:   Individual Values
    actual_range:  [-1.8     42.32636]
    valid_range:   [-1.8 45. ]
------------ TIME -----------------------------
<xarray.DataArray 'time' (time: 2064)> Size: 17kB
array(['1854-01-01T00:00:00.000000000', '1854-02-01T00:00:00.000000000',
       '1854-03-01T00:00:00.000000000', ..., '2025-10-01T00:00:00.00000000

In [ ]:
# Guardar en .npz (equivalente al NetCDF)
sst_ma = np.ma.masked_invalid(sst.values)

np.savez_compressed(
    "sst_2deg_1854_2025_xarray.npz",
    sst_data=sst_ma.data,
    sst_mask=sst_ma.mask,
    time=time_sst.values,
    lat=lat.values,
    lon=lon.values
)

In [33]:
# Cargar después (y reconstruir con xarray)
data = np.load("sst_2deg_1854_2025_xarray.npz")

sst_ma = np.ma.MaskedArray(
    data["sst_data"],
    mask=data["sst_mask"]
)

da_sst = xr.DataArray(
    sst_ma,
    dims=("time", "lat", "lon"),
    coords={
        "time": data["time"],
        "lat": data["lat"],
        "lon": data["lon"],
    },
    name="sst"
)

da_sst

<xarray.DataArray 'sst' (time: 2064, lat: 89, lon: 180)> Size: 132MB
array([[[-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
        [-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
        [-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
        ...,
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan]],

       [[-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
        [-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
        [-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
...
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan]],

       [[-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
        [-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
        [-1.8      , -1.8      , -1.8      , ..., -1.8      ,
         -1.8      , -1.8      ],
        ...,
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [       nan,        nan,        nan, ...,        nan,
                nan,        nan]]], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 17kB 1854-01-01 1854-02-01 ... 2025-12-01
  * lat      (lat) float32 356B 88.0 86.0 84.0 82.0 ... -82.0 -84.0 -86.0 -88.0
  * lon      (lon) float32 720B 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0